In [ ]:
# fetching data from the website (data fetching automation)
from pathlib import Path
import requests 

def download_one_file_of_raw_data (year:int, month:int) -> Path:
    URL = f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month:02d}.parquet'
    response=requests.get(URL)

    if response.status_code==200:
        path = f'../data/raw/rides_{year}-{month:02d}.parquet'
        open(path,'wb').write(response.content)
        return path
    else:
        raise Exception(f'{URL} is not available') 

In [2]:
download_one_file_of_raw_data (year=2026, month=7)

'../data/raw/rides_2026-07.parquet'

In [4]:
import pandas as pd
rides= pd.read_parquet('../data/raw/rides_2026-07.parquet')
rides.head(5)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,request_source
0,1,2026-07-01 00:36:25,2026-07-01 00:43:18,1.0,2.00,1.0,N,138,138,2,...,8.00,0.5,0.00,0.00,1.0,19.50,0.0,2.0,0.00,NaN
1,1,2026-07-01 00:02:53,2026-07-01 00:17:12,1.0,3.20,1.0,N,162,112,2,...,4.25,0.5,0.00,0.00,1.0,21.35,2.5,0.0,0.75,NaN
2,2,2026-07-01 00:18:56,2026-07-01 00:27:38,1.0,3.18,1.0,N,137,263,1,...,1.00,0.5,4.81,0.00,1.0,24.06,2.5,0.0,0.75,NaN
3,1,2026-07-01 00:35:06,2026-07-01 00:57:51,0.0,10.00,1.0,N,138,246,1,...,11.25,0.5,12.30,7.46,1.0,74.01,2.5,2.0,0.75,NaN
4,7,2026-07-01 00:36:59,2026-07-01 00:36:59,2.0,9.62,1.0,N,140,106,1,...,0.00,0.5,9.59,0.00,1.0,57.54,2.5,0.0,0.75,NaN


In [5]:
# for our business problem we need two columns
rides = rides[[
    'tpep_pickup_datetime', 'PULocationID'
]]

In [6]:
rides.rename(columns={'tpep_pickup_datetime':'pickup_datetime',
'PULocationID':'pickup_location_id'}, inplace= True)

In [7]:
# Validating data

rides['pickup_datetime'].describe()

count                       3530109
mean     2026-07-17 04:05:43.459226
min             2008-12-30 23:06:00
25%             2026-07-09 20:38:49
50%             2026-07-17 10:21:03
75%             2026-07-24 15:44:31
max             2026-08-05 20:54:00
Name: pickup_datetime, dtype: object

In [ ]:
# ensuring the trips actually took place in July 2026 (Data Validation)
rides= rides[rides['pickup_datetime'] >= '2026-07-01']
rides= rides[rides['pickup_datetime'] < '2026-08-01']
rides['pickup_datetime'].describe()

count                       3530063
mean     2026-07-17 04:08:06.937820
min             2026-07-01 00:00:00
25%             2026-07-09 20:38:47
50%             2026-07-17 10:20:56
75%             2026-07-24 15:44:19
max             2026-07-31 23:59:59
Name: pickup_datetime, dtype: object

In [9]:
rides.to_parquet('../data/transformed/validated_rides_2026_07.parquet')